# Decoding sweep — free GLEU, no retraining

The model scored **59.29 GLEU** on JFLEG test with plain greedy decoding, and
its `eval_loss` had plateaued, so a second epoch on the same data buys nothing.
Two measurements point at decoding instead:

- the model edits **79.0%** of JFLEG sentences while the human annotators edit
  **86.3%** — it under-corrects,
- held-out dev recall is **0.301** against precision **0.474** — same story.

Beam search and a length penalty both push a sequence model towards committing
to edits. Everything here is tuned on **validation** and reported on **test**,
so the headline number is never chosen by looking at test.

In [ ]:
import os, re, json, math, glob, time, itertools
from collections import Counter
import pandas as pd, torch

T0=time.time()
def stage(m): print(f"[stage +{time.time()-T0:7.1f}s] {m}", flush=True)
def norm(s): return re.sub(r"\s+", " ", s).strip()

def find_dir(pat):
    hits = glob.glob(pat, recursive=True)
    return hits[0] if hits else None

MODEL_DIR = find_dir("/kaggle/input/**/gec/saves/qwen3-0.6b-gec")
print("model:", MODEL_DIR)
assert MODEL_DIR, "trained model not found in the attached kernel output"

In [ ]:
def ngrams(t, n):
    return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1))

def gleu_corpus(sources, hyps, refs_list, N=4):
    n_refs = min(len(r) for r in refs_list)
    per_ref = []
    for j in range(n_refs):
        num=[0]*(N+1); den=[0]*(N+1); hl=rl=0
        for src, hyp, refs in zip(sources, hyps, refs_list):
            s,h,r = src.split(), hyp.split(), refs[j].split()
            hl += len(h); rl += len(r)
            for n in range(1, N+1):
                hg, sg, rg = ngrams(h,n), ngrams(s,n), ngrams(r,n)
                hit = sum((hg & rg).values())
                kept_wrong = sum((hg & sg).values()) - sum((hg & sg & rg).values())
                num[n] += max(hit - kept_wrong, 0); den[n] += sum(hg.values())
        logs=[]
        for n in range(1, N+1):
            p=(num[n]/den[n]) if den[n] else 0.0
            logs.append(math.log(p) if p>0 else -9e9)
        bp = 1.0 if hl>rl else math.exp(1-rl/max(hl,1))
        per_ref.append(bp*math.exp(sum(logs)/N))
    return 100*sum(per_ref)/len(per_ref)

def retokenize(s):
    s = re.sub(r"\s*([,;:!?])", r" \1", s)
    s = re.sub(r"(?<!\d)\s*\.(?!\d)", r" .", s)
    s = re.sub(r"\s*'\s*(s|t|re|ve|ll|d|m)\b", r" '\1", s)
    return norm(s)

from datasets import load_dataset
stage("loading JFLEG + verifying the metric")
JDATA={}
for split in ("validation","test"):
    ds = load_dataset("jhu-clsp/jfleg", split=split)
    src=[s.strip() for s in ds["sentence"]]
    refs=[[r.strip() for r in c] for c in ds["corrections"]]
    keep=[i for i,s in enumerate(src) if s]
    JDATA[split]=([src[i] for i in keep],[refs[i] for i in keep])
    base=gleu_corpus(JDATA[split][0], JDATA[split][0], JDATA[split][1])
    pub={"test":40.54,"validation":38.21}[split]
    print(f"{split:<11} source={base:6.2f} (published {pub})")
    assert abs(base-pub)<1.5, "GLEU disagrees with the published baseline"
print("metric verified")

In [ ]:
stage("loading model")
from transformers import AutoTokenizer, AutoModelForCausalLM

INSTRUCTION = (
    "Correct the grammar, spelling, and punctuation of the sentence. "
    "Do not change its meaning or style. "
    "Reply with the corrected sentence only. "
    "If it is already correct, repeat it unchanged."
)

tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, dtype=torch.float16,
    device_map={"": 0} if torch.cuda.is_available() else None)
model.eval()
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"

@torch.no_grad()
def correct(texts, bs=32, **gen_kwargs):
    out=[]
    for i in range(0, len(texts), bs):
        chunk=texts[i:i+bs]
        prompts=[tok.apply_chat_template(
            [{"role":"user","content": INSTRUCTION + "\n" + t}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False)
            for t in chunk]
        enc=tok(prompts, return_tensors="pt", padding=True,
                truncation=True, max_length=512).to(model.device)
        gen=model.generate(**enc, max_new_tokens=160, do_sample=False,
                           pad_token_id=tok.pad_token_id, **gen_kwargs)
        out += [norm(s) for s in tok.batch_decode(
            gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
    return out

In [ ]:
stage("sweeping decoding configs on VALIDATION")

CONFIGS = [
    ("greedy",                 {}),
    ("beam4",                  {"num_beams": 4}),
    ("beam4 lp=1.2",           {"num_beams": 4, "length_penalty": 1.2}),
    ("beam4 lp=0.8",           {"num_beams": 4, "length_penalty": 0.8}),
    ("beam8",                  {"num_beams": 8}),
    ("beam4 no-repeat-3",      {"num_beams": 4, "no_repeat_ngram_size": 3}),
]

src_v, refs_v = JDATA["validation"]
rows=[]
for name, kw in CONFIGS:
    t0=time.time()
    hyp = correct(src_v, bs=16 if kw.get("num_beams",1) > 4 else 32, **kw)
    g = gleu_corpus(src_v, [retokenize(h) for h in hyp], refs_v)
    edited = 100*sum(1 for a,b in zip(src_v,hyp) if norm(a)!=b)/len(src_v)
    rows.append({"config": name, "GLEU": g, "edited %": edited, "sec": time.time()-t0})
    print(f"  {name:<22} GLEU={g:6.2f}  edited={edited:5.1f}%  ({time.time()-t0:.0f}s)")

dfv = pd.DataFrame(rows).sort_values("GLEU", ascending=False)
print("\n" + "="*70)
print("VALIDATION sweep (humans edit 86.3% of sentences)")
print("="*70)
print(dfv.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

BEST_NAME, BEST_KW = max(zip([r["config"] for r in rows], [k for _,k in CONFIGS],
                             [r["GLEU"] for r in rows]), key=lambda x: x[2])[:2]
print("\nselected on validation:", BEST_NAME, BEST_KW)

In [ ]:
stage("reporting on TEST with the validation-selected config")

src_t, refs_t = JDATA["test"]
greedy_t = correct(src_t)
best_t   = correct(src_t, bs=16 if BEST_KW.get("num_beams",1) > 4 else 32, **BEST_KW)

res = {
    "source (no correction)": gleu_corpus(src_t, src_t, refs_t),
    "coedit-large (measured earlier)": 55.60,
    "qwen3-ft greedy": gleu_corpus(src_t, [retokenize(h) for h in greedy_t], refs_t),
    f"qwen3-ft {BEST_NAME}": gleu_corpus(src_t, [retokenize(h) for h in best_t], refs_t),
    "human reference": gleu_corpus(src_t, [r[0] for r in refs_t], [r[1:] for r in refs_t]),
}

print("\n" + "="*70)
print("JFLEG TEST — GLEU")
print("="*70)
for k, v in res.items():
    print(f"{k:<34}{v:>8.2f}")

ed = 100*sum(1 for a,b in zip(src_t,best_t) if norm(a)!=b)/len(src_t)
print(f"\n{BEST_NAME} edited {ed:.1f}% of sentences (humans: 86.3%)")

pd.DataFrame({"src": src_t, "greedy": greedy_t, BEST_NAME: best_t,
              "ref1": [r[0] for r in refs_t]}).to_csv("decoding_predictions.csv", index=False)
print("saved decoding_predictions.csv")